In [2]:
import accelerate
!pip install -q \
transformers == 4.56
.1 \
    accelerate == 1.10
.1 \
    peft == 0.17
.1 \
    bitsandbytes == 0.47
.0 \
    datasets \
    pyarrow \
    gdown \
    pillow \
    requests

In [4]:
import torch

print("GPUs:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))

GPUs: 1
0 Tesla T4


In [2]:
MODEL_ID = "google/medgemma-4b-it"
DATASET_FILE = "dataset.parquet"

TRAIN_SIZE = 1000
VAL_SIZE = 100

EPOCHS = 1

LEARNING_RATE = 2e-4
WEIGHT_DECAY = 0.01

GRADIENT_ACCUMULATION = 4

LORA_R = 16
LORA_ALPHA = 16
LORA_DROPOUT = 0.05

CACHE_DIR = Path("./cache")
CACHE_DIR.mkdir(exist_ok=True)

MAX_DOWNLOAD_WORKERS = 4
REQUEST_TIMEOUT = 30

OUTPUT_DIR = Path("./outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(exist_ok=True)

FINAL_DIR = OUTPUT_DIR / "final"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

TORCH_DTYPE = torch.float16

USE_4BIT = True

SEED = 42

print("Device:", DEVICE)
print("Model:", MODEL_ID)

NameError: name 'Path' is not defined

In [6]:
import os

HF_TOKEN = os.environ["HF_TOKEN"]
REDIVIS_API_KEY = os.environ["REDIVIS_API_KEY"]

In [7]:
random.seed(SEED)

torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [8]:
import gdown

FILE_ID = "1Dv5gDCdJCa0Gv3jbifsbghdbZABdYsML"

if not os.path.exists(DATASET_FILE):
    print("Downloading dataset...")

    gdown.download(
        id=FILE_ID,
        output=DATASET_FILE,
        quiet=False,
    )

Downloading...
From (original): https://drive.google.com/uc?id=1Dv5gDCdJCa0Gv3jbifsbghdbZABdYsML
From (redirected): https://drive.google.com/uc?id=1Dv5gDCdJCa0Gv3jbifsbghdbZABdYsML&confirm=t&uuid=241d62f3-cb78-4aa0-91e6-5b102d39d606
To: /content/dataset.parquet
100%|██████████| 174M/174M [00:10<00:00, 15.9MB/s] 


In [9]:
df = pd.read_parquet(DATASET_FILE)

df.shape

(223445, 4)

In [10]:
df.head()

,report,split,prompt,image_file_ids
0,NARRATIVE:\nRADIOGRAPHIC EXAMINATION OF THE CH...,train,You are an expert thoracic radiologist.\n\n ...,[s6cj-f5e10m11h.4MafEl-RrYhHLi2oPR7rSQ]
1,NARRATIVE:\nFRONTAL AND LATERAL CHEST RADIOGRA...,train,You are an expert thoracic radiologist.\n\n ...,[s6cj-f5e10m11h.Oc5LOGuATXu630N5amOXmw]
2,NARRATIVE:\nFRONTAL AND LATERAL CHEST RADIOGRA...,train,You are an expert thoracic radiologist.\n\n ...,"[s6cj-f5e10m11h.Oc5LOGuATXu630N5amOXmw, s6cj-f..."
3,"NARRATIVE:\nSINGLE VIEW OF THE CHEST, 4 VIEWS ...",train,You are an expert thoracic radiologist.\n\n ...,"[s6cj-f5e10m11h.Oc5LOGuATXu630N5amOXmw, s6cj-f..."
4,"NARRATIVE:\nCHEST, ONE VIEW: 2-10-2001\nFINDIN...",train,You are an expert thoracic radiologist.\n\n ...,[s6cj-f5e10m11h.kbtoHXjniUchFJPuxvqx3Q]


In [11]:
shuffled = df.sample(
    frac=1,
    random_state=SEED
).reset_index(drop=True)

train_df = shuffled.iloc[
    :TRAIN_SIZE
].reset_index(drop=True)

val_df = shuffled.iloc[
    TRAIN_SIZE:
    TRAIN_SIZE + VAL_SIZE
].reset_index(drop=True)

print("Training:", len(train_df))
print("Validation:", len(val_df))

Training: 1000
Validation: 100


In [12]:
BASE_URL = "https://redivis.com/api/v1/rawFiles"

HEADERS = {
    "Authorization": f"Bearer {REDIVIS_API_KEY}"
}

In [13]:
def download_image(file_id):
    cache_path = CACHE_DIR / f"{file_id}.png"

    if cache_path.exists():
        return str(cache_path)

    response = requests.get(
        f"{BASE_URL}/{file_id}",
        headers=HEADERS,
        timeout=REQUEST_TIMEOUT,
    )

    response.raise_for_status()

    image = Image.open(
        BytesIO(response.content)
    ).convert("RGB")

    image.save(cache_path)

    return str(cache_path)

In [14]:
def download_images(file_ids):
    with ThreadPoolExecutor(
            max_workers=MAX_DOWNLOAD_WORKERS
    ) as executor:
        paths = list(
            executor.map(
                download_image,
                file_ids
            )
        )

    return paths

In [15]:
def load_images(paths):
    return [
        Image.open(path).convert("RGB")
        for path in paths
    ]

In [16]:
MAX_IMAGES = 10


def get_images(file_ids):
    file_ids = list(file_ids)[:MAX_IMAGES]

    paths = download_images(file_ids)

    images = load_images(paths)

    return images

In [36]:
def get_example(row):
    file_ids = list(row["image_file_ids"])

    return [
        {
            "role": "user",
            "content": (
                    [
                        {
                            "type": "image",
                            "image": image
                        }
                        for image in images
                    ]
                    +
                    [
                        {
                            "type": "text",
                            "text": row["prompt"]
                        }
                    ]
            ),
        },
        {
            "role": "assistant",
            "content": [
                {
                    "type": "text",
                    "text": row["report"]
                }
            ],
        },
    ]

In [18]:
example = get_example(train_df.iloc[4])

print(example[0]["role"])
print(example[1]["role"])
print("Images:", len([
    x for x in example[0]["content"]
    if x["type"] == "image"
]))

user
assistant
Images: 23


In [19]:
from huggingface_hub import login

login(token=HF_TOKEN)

In [20]:
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
)

processor.tokenizer.padding_side = "right"

print("Processor loaded.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

...

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

Processor loaded.


In [21]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_storage=torch.float16,
)

In [22]:
def load_model(local_rank):
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        torch_dtype=torch.float16,
        device_map={"":  local_rank},
        attn_implementation="sdpa",
        token=HF_TOKEN,
    )

    return model

config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.96G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.64G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:212: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

In [23]:
def prepare_model(model):

    model = prepare_model_for_kbit_training(model)

    model.gradient_checkpointing_enable()

    model.enable_input_require_grads()

    model.config.use_cache = False

    model = get_peft_model(
        model,
        get_lora_config()
    )

    return model

In [ ]:
from accelerate import Accelerator
from accelerate import notebook_launcher

In [42]:
def prepare_batch(example):
    text = processor.apply_chat_template(
        example,
        tokenize=False,
        add_generation_prompt=False,
    )

    images = [
        item["image"]
        for item in example[0]["content"]
        if item["type"] == "image"
    ]

    images = images[:MAX_IMAGES]

    batch = processor(
        text=[text],
        images=[images],
        return_tensors="pt",
    )

    device = get_input_device(model)

    batch = {
        key: value.to(device, non_blocking=True)
        for key, value in batch.items()
    }

    labels = batch["input_ids"].clone()

    pad_id = processor.tokenizer.pad_token_id

    if pad_id is not None:
        labels[labels == pad_id] = -100

    try:
        boi_id = processor.tokenizer.convert_tokens_to_ids(
            processor.tokenizer.special_tokens_map["boi_token"]
        )

        labels[labels == boi_id] = -100

    except Exception:
        pass

    labels[labels == 262144] = -100

    batch["labels"] = labels

    return batch

In [ ]:
def test_worker():

    accelerator = Accelerator(
        mixed_precision="fp16",
        gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    )

    accelerator.print(
        f"Process {accelerator.process_index} "
        f"using GPU {accelerator.local_process_index}"
    )

    accelerator.print(
        f"World size: {accelerator.num_processes}"
    )

In [24]:
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    target_modules="all-linear",
    task_type="CAUSAL_LM",
)

model = get_peft_model(
    model,
    lora_config
)

model.print_trainable_parameters()

trainable params: 38,497,792 || all params: 4,338,577,264 || trainable%: 0.8873


In [41]:
def get_input_device(model):
    return model.get_input_embeddings().weight.device

In [26]:
example = get_example(
    train_df.iloc[0]
)

batch = prepare_batch(example)

for key, value in batch.items():
    print(
        key,
        value.shape if hasattr(value, "shape") else type(value)
    )

input_ids torch.Size([1, 560])
attention_mask torch.Size([1, 560])
token_type_ids torch.Size([1, 560])
pixel_values torch.Size([1, 3, 896, 896])
labels torch.Size([1, 560])


In [27]:
optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

In [28]:
model.train()

outputs = model(**batch)

loss = outputs.loss

print(loss.item())

/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


2.7688441276550293


In [29]:
optimizer.zero_grad(
    set_to_none=True
)

loss.backward()

In [30]:
optimizer.step()

optimizer.zero_grad(
    set_to_none=True
)

In [31]:
model.print_trainable_parameters()

trainable params: 38,497,792 || all params: 4,338,577,264 || trainable%: 0.8873


In [32]:
import torch

print(f"Allocated : {torch.cuda.memory_allocated() / 1024 ** 3:.2f} GB")
print(f"Reserved  : {torch.cuda.memory_reserved() / 1024 ** 3:.2f} GB")
print(f"Peak      : {torch.cuda.max_memory_allocated() / 1024 ** 3:.2f} GB")

Allocated : 5.23 GB
Reserved  : 7.77 GB
Peak      : 6.82 GB


In [39]:
import time
import torch
import gc


def benchmark_steps(num_steps=10):
    model.train()

    times = []

    for step in range(num_steps):
        example = get_example(
            train_df.iloc[step]
        )

        batch = prepare_batch(example)

        torch.cuda.synchronize()
        start = time.perf_counter()

        optimizer.zero_grad(set_to_none=True)

        outputs = model(**batch)

        loss = outputs.loss

        loss.backward()

        optimizer.step()

        torch.cuda.synchronize()
        elapsed = time.perf_counter() - start

        times.append(elapsed)

        print(
            f"Step {step + 1}/{num_steps} | "
            f"Loss {loss.item():.4f} | "
            f"{elapsed:.2f}s"
        )

        del batch, outputs, loss

    avg = sum(times) / len(times)

    print("\n==============================")
    print(f"Average step: {avg:.2f}s")
    print(f"Examples/hour: {3600 / avg:.1f}")
    print("==============================")

    gc.collect()
    torch.cuda.empty_cache()


benchmark_steps(10)

/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


Step 1/10 | Loss 1.5531 | 4.55s


OutOfMemoryError: CUDA out of memory. Tried to allocate 1.94 GiB. GPU 0 has a total capacity of 14.56 GiB of which 493.81 MiB is free. Including non-PyTorch memory, this process has 14.08 GiB memory in use. Of the allocated memory 12.57 GiB is allocated by PyTorch, and 1.37 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [40]:
import torch

print("GPUs:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))

GPUs: 1
0 Tesla T4


In [38]:
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

example = get_example(train_df.iloc[0])
batch = prepare_batch(example)

print("Images:",
      len([x for x in example[0]["content"] if x["type"] == "image"]))

print("Input IDs:", batch["input_ids"].shape)
print("Pixels:", batch["pixel_values"].shape)

model.train()

optimizer.zero_grad(set_to_none=True)

outputs = model(**batch)

print("After forward:",
      torch.cuda.memory_allocated() / 1024 ** 3)

loss = outputs.loss
loss.backward()

print("After backward:",
      torch.cuda.memory_allocated() / 1024 ** 3)

print("Peak:",
      torch.cuda.max_memory_allocated() / 1024 ** 3)

optimizer.step()

Images: 1
Input IDs: torch.Size([1, 560])
Pixels: torch.Size([1, 3, 896, 896])


/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


After forward: 10.608348846435547
After backward: 9.977491855621338
Peak: 11.709171295166016


In [37]:
for i in range(30):
    example = get_example(train_df.iloc[i])

    n_images = len([
        x for x in example[0]["content"]
        if x["type"] == "image"
    ])

    batch = prepare_batch(example)

    print(
        f"{i:3d} | "
        f"images={n_images} | "
        f"tokens={batch['input_ids'].shape[1]} | "
        f"pixels={tuple(batch['pixel_values'].shape)}"
    )

    del example, batch
    gc.collect()
    torch.cuda.empty_cache()

  0 | images=1 | tokens=560 | pixels=(1, 3, 896, 896)
  1 | images=4 | tokens=1990 | pixels=(4, 3, 896, 896)
  2 | images=4 | tokens=2062 | pixels=(4, 3, 896, 896)
  3 | images=1 | tokens=465 | pixels=(1, 3, 896, 896)
  4 | images=10 | tokens=8627 | pixels=(10, 3, 896, 896)
  5 | images=1 | tokens=667 | pixels=(1, 3, 896, 896)
  6 | images=10 | tokens=4082 | pixels=(10, 3, 896, 896)
  7 | images=2 | tokens=937 | pixels=(2, 3, 896, 896)
  8 | images=10 | tokens=15309 | pixels=(10, 3, 896, 896)
  9 | images=7 | tokens=2940 | pixels=(7, 3, 896, 896)


KeyboardInterrupt: 

In [44]:
print(model.hf_device_map)

{'': 0}


In [45]:
!pip install -q accelerate

In [46]:
from accelerate import Accelerator

In [ ]:
import torch

print(torch.cuda.device_count())

assert torch.cuda.device_count() == 2, \
    "Kaggle must show exactly 2 GPUs"

for i in range(2):
    print(i, torch.cuda.get_device_name(i))

In [1]:
from accelerate import Accelerator


def train_worker():
    accelerator = Accelerator(
        gradient_accumulation_steps=GRADIENT_ACCUMULATION,
        mixed_precision="fp16"
    )

    device = accelerator.device

    accelerator.print("Process started")
    accelerator.print("World size:", accelerator.num_processes)
    accelerator.print("Device:", device)

    processor = AutoProcessor.from_pretrained(
        MODEL_ID
    )

    processor.tokenizer.padding_side = "right"

    processor = AutoProcessor.from_pretrained(
        MODEL_ID
    )

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_storage=torch.float16,
    )

    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        torch_dtype=torch.float16,
        device_map={"": accelerator.local_process_index},
        attn_implementation="sdpa",
    )

    model = prepare_model_for_kbit_training(model)

    model.gradient_checkpointing_enable()
    model.enable_input_require_grads()
    model.config.use_cache = False

    lora_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        target_modules="all-linear",
        task_type="CAUSAL_LM",
    )

    model = get_peft_model(
        model,
        lora_config
    )

    accelerator.print(
        "Trainable parameters:"
    )
    model.print_trainable_parameters()

    optimizer = AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    model, optimizer = accelerator.prepare(
        model,
        optimizer
    )

    accelerator.print("Distributed model ready")

    return accelerator, model, processor, optimizer

In [2]:
from accelerate import notebook_launcher

notebook_launcher(
    train_worker,
    num_processes=2
)

NameError: name 'train_worker' is not defined

In [1]:
!git status

On branch master
Your branch is up to date with 'origin/master'.

Changes not staged for commit:
  (use "git add/rm <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	deleted:    ../api/requirements

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	../api/requirements.txt

no changes added to commit (use "git add" and/or "git commit -a")
